# TP3 — Évaluation : mesurer pour choisir

Depuis ce matin, on a fait des choix : découpage, modèle d'embeddings, `k`, recherche sémantique ou hybride, prompt,
modèle de génération… Et on les a jugés sur trois questions. Ce n'est pas une mesure.

Comment comparer objectivement deux variantes ? Il faut deux ingrédients :

- un **jeu de test** : des questions dont on connaît la réponse ;
- une **métrique** : un chiffre qui dit si le système a trouvé la réponse.

Le parcours de base montre **une** manière d'évaluer le système.
Les bonus ajoutent d'autres méthodes, d'autres métriques, puis l'évaluation des réponses du LLM.

Plan :
1. Un jeu de questions
2. Évaluer la recherche sémantique avec le hit@k

Pour aller plus loin (bonus, dans l'ordre de votre choix) :
- Bonus 1 — une deuxième métrique : le MRR
- Bonus 2 — une deuxième méthode : la recherche par mots-clés (BM25)
- Bonus 3 — comparer deux découpages
- Bonus 4 — mesurer vos améliorations du TP1
- Bonus 5 — faire générer des questions par le LLM
- Bonus 6 — évaluer la génération avec un LLM juge
- Bonus 7 — un juge fiable ?
- Bonus 8 — les questions pièges

In [ ]:
import json
import re

import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from litellm import completion

import settings
from rag import load_index, load_pdfs, chunk_pages, embed, search, search_bm25, answer

load_dotenv()
chunks, embeddings = load_index()

## 1. Un jeu de questions

Une question de test = `{"question", "source", "pages"}` : la réponse se trouve à l'une de ces pages de ce document
(une même information figure souvent à plusieurs endroits : résumé, tableau, texte).
On repère la réponse par sa **page** et non par un numéro de chunk : le jeu reste valable si on change le découpage.

Le fichier `eval/questions.json` contient une vingtaine de questions écrites à la main et vérifiées.

In [ ]:
with open("eval/questions.json", encoding="utf-8") as f:
    jeu_test = json.load(f)

print(len(jeu_test), "questions")
for item in jeu_test[:5]:
    print(f"- {item['question']}  →  {item['source']} p.{item['pages']}")

## 2. Évaluer la recherche sémantique avec le hit@k

On prend **une méthode**, la recherche sémantique du TP1 (`search`), et **une métrique**, le hit@k.

Pour chaque question, on lance la recherche et on note **à quel rang** apparaît la bonne page.
Le **hit@k** est la proportion de questions dont la bonne page figure dans les k premiers résultats.
Exemple : hit@5 = 0,8 signifie que, dans 80 % des cas, le LLM aura la bonne page sous les yeux avec `k = 5`.

In [ ]:
def rang_bonne_page(item, resultats_recherche):
    """Rang (à partir de 1) du premier chunk issu de la bonne page ; None si elle est absente."""
    for rang, r in enumerate(resultats_recherche, start=1):
        if r["source"] == item["source"] and r["page"] in item["pages"]:
            return rang
    return None


def evaluer_retrieval(fonction_recherche, jeu_test, k_max=10):
    """Lance la recherche sur chaque question et renvoie la liste des rangs de la bonne page.

    fonction_recherche(question, k) -> liste de chunks
    """
    return [rang_bonne_page(item, fonction_recherche(item["question"], k_max)) for item in jeu_test]


def hit_at_k(rangs, k):
    """Proportion de questions dont la bonne page est dans les k premiers résultats."""
    return float(np.mean([rang is not None and rang <= k for rang in rangs]))


semantique = lambda q, k: search(q, chunks, embeddings, k)
rangs = evaluer_retrieval(semantique, jeu_test)

for item, rang in zip(jeu_test, rangs):
    print(f"rang {rang or '-':>2}  {item['question']}")
print(f"\nhit@1 = {hit_at_k(rangs, 1):.2f}   hit@5 = {hit_at_k(rangs, 5):.2f}   hit@10 = {hit_at_k(rangs, 10):.2f}")

In [ ]:
def afficher_resultats(resultats):
    """Tableau et courbes hit@k de toutes les variantes mesurées (resultats = {nom: rangs})."""
    for nom, rangs_variante in resultats.items():
        print(f"{nom:<28} hit@1={hit_at_k(rangs_variante, 1):.2f}  hit@5={hit_at_k(rangs_variante, 5):.2f}  "
              f"hit@10={hit_at_k(rangs_variante, 10):.2f}")
        plt.plot(range(1, 11), [hit_at_k(rangs_variante, k) for k in range(1, 11)], marker="o", label=nom)
    plt.xlabel("k"); plt.ylabel("hit@k"); plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.legend()
    plt.title("La bonne page est-elle dans les k premiers résultats ?")
    plt.show()


resultats = {"sémantique": rangs}  # les bonus y ajoutent leurs variantes
afficher_resultats(resultats)

**À observer**

- La courbe monte avec k : plus on donne de chunks au LLM, plus la bonne page a de chances d'y être.
  En contrepartie, chaque chunk coûte des tokens.
- Le point où la courbe plafonne aide à choisir `settings.TOP_K`.
- Quelles questions échouent (rang `-`) ? Identifiant exact, question en français sur une datasheet en anglais,
  petit document noyé parmi les gros… Ce sont autant de pistes d'amélioration.

**Vous avez une évaluation complète** : un jeu de test, une méthode, une métrique, un chiffre.
Pour juger un réglage, on le change, on relance ces deux cellules et on compare le hit@5.
Les bonus ci-dessous approfondissent chaque ingrédient.

## Pour aller plus loin (bonus)

Les bonus sont indépendants : faites-les dans l'ordre de votre choix. Seul le bonus 7 demande d'avoir exécuté le bonus 6.
Les bonus 1 à 4 tournent sans clé d'API ; les bonus 5 à 8 appellent le LLM.

### Bonus 1 — Une deuxième métrique : le MRR

Le hit@k répond par oui ou par non : la bonne page est-elle dans le top k ? Il ne distingue pas une bonne page
au rang 1 d'une bonne page au rang 5.

Le **MRR** (*Mean Reciprocal Rank*, rang réciproque moyen) fait cette différence. C'est la moyenne de 1/rang de la
bonne page, avec 0 si elle est absente : rang 1 → 1 ; rang 2 → 0,5 ; rang 5 → 0,2.
Plus le MRR est proche de 1, plus la bonne page arrive haut.

In [ ]:
def mrr(rangs):
    """Moyenne de 1/rang de la bonne page (0 quand elle est absente)."""
    return float(np.mean([1 / rang if rang else 0 for rang in rangs]))


for nom, rangs_variante in resultats.items():  # toutes les variantes mesurées jusqu'ici
    print(f"{nom:<28} hit@5={hit_at_k(rangs_variante, 5):.2f}  MRR={mrr(rangs_variante):.2f}")

**À observer**

Deux variantes peuvent avoir le même hit@5 et des MRR différents. Celle qui place la bonne page plus haut
permet de réduire `k` sans la perdre, donc d'envoyer moins de tokens au LLM.
Relancez cette cellule après les autres bonus pour comparer toutes vos variantes.

### Bonus 2 — Une deuxième méthode : la recherche par mots-clés (BM25)

Même jeu de test, même métrique, autre méthode : `search_bm25` du TP1. On l'ajoute au tableau `resultats`
et on compare les courbes.

In [ ]:
mots_cles = lambda q, k: search_bm25(q, chunks, k)
resultats["BM25"] = evaluer_retrieval(mots_cles, jeu_test)
afficher_resultats(resultats)

**À observer**

- Laquelle gagne sur ce jeu ? Cela dépend des questions (identifiants exacts ou paraphrases), de la langue et du
  découpage. D'où l'intérêt d'un jeu de test **représentatif des vraies questions des utilisateurs**.
- Rappel du TP1 : ce BM25 compte aussi les mots vides (`de`, `la`, `est`…). Le bonus 4 mesure la version sans mots vides.

### Bonus 3 — Comparer deux découpages

Même méthode pour le découpage : on reconstruit un index avec d'autres paramètres, et on mesure.
Ici, des chunks deux fois plus grands (300 mots au lieu de 150). Le calcul des embeddings prend 1 à 2 minutes.

In [ ]:
pages = load_pdfs()
chunks_300 = chunk_pages(pages, chunk_size=300, overlap=50)
embeddings_300 = embed([c["text"] for c in chunks_300])
print(len(chunks_300), "chunks de 300 mots (contre", len(chunks), "de 150)\n")

resultats["sémantique, chunks 300"] = evaluer_retrieval(lambda q, k: search(q, chunks_300, embeddings_300, k), jeu_test)
afficher_resultats(resultats)

**À observer**

Des chunks plus grands contiennent plus souvent la réponse (moins de coupures), mais ils sont moins précis et coûtent
plus de tokens. Il n'y a pas de bonne taille universelle : elle dépend des documents et des questions.
**On la choisit en mesurant.**

### Bonus 4 — Mesurer vos améliorations du TP1

BM25 sans mots vides, recherche hybride, découpage par paragraphes : chaque variante s'ajoute au tableau `resultats`
et se compare à la base. Quelle combinaison gagne ? Comparez vos chiffres entre groupes.

La cellule importe les fonctions du **corrigé du TP1** (`solutions/tp1.py`, fourni en fin de TP1) : pas besoin d'avoir
terminé le TP1 pour faire ce bonus. Vous préférez vos propres fonctions ? Collez-les sous l'import : elles remplacent
celles du corrigé.

L'index par paragraphes est rechargé depuis `index_paragraphes/` s'il a été sauvegardé au TP1, recalculé sinon
(2 à 3 min).

In [ ]:
from pathlib import Path

try:
    from solutions.tp1 import search_bm25_sans_mots_vides, search_hybrid, chunk_pages_par_paragraphes
except ModuleNotFoundError:
    print("Corrigé du TP1 absent (solutions/tp1.py) : collez vos fonctions ci-dessous.")

# Collez ici vos fonctions du TP1 si vous préférez les vôtres : MOTS_VIDES, tokenize_sans_mots_vides,
# search_bm25_sans_mots_vides, search_hybrid, chunk_pages_par_paragraphes.


# -- BM25 sans mots vides, puis recherche hybride --
resultats["BM25 sans mots vides"] = evaluer_retrieval(lambda q, k: search_bm25_sans_mots_vides(q, chunks, k), jeu_test)
resultats["hybride (RRF)"] = evaluer_retrieval(lambda q, k: search_hybrid(q, chunks, embeddings, k), jeu_test)

# -- découpage par paragraphes : index sauvegardé au TP1 (exercice 2), sinon recalculé --
if Path("index_paragraphes").exists():
    chunks_paragraphes, embeddings_paragraphes = load_index("index_paragraphes")
else:
    chunks_paragraphes = chunk_pages_par_paragraphes(load_pdfs())
    embeddings_paragraphes = embed([c["text"] for c in chunks_paragraphes])  # 2 à 3 min
resultats["sémantique, par paragraphes"] = evaluer_retrieval(
    lambda q, k: search(q, chunks_paragraphes, embeddings_paragraphes, k), jeu_test)
resultats["hybride + par paragraphes"] = evaluer_retrieval(
    lambda q, k: search_hybrid(q, chunks_paragraphes, embeddings_paragraphes, k), jeu_test)

afficher_resultats(resultats)

### Bonus 5 — Faire générer des questions par le LLM

Écrire des questions à la main prend du temps. On peut en faire **générer** par le LLM : on tire des chunks au hasard
et on lui demande, pour chacun, une question à laquelle le chunk répond. La vérité terrain est la page du chunk.

Comptez une vingtaine d'appels à l'API. Regardez les questions produites : certaines seront mauvaises (chunk de tableau
illisible, question ambiguë). La qualité du jeu de test conditionne toute la mesure : en pratique, on relit et on trie.

In [ ]:
import random

PROMPT_QUESTION = """Voici un extrait d'une datasheet de composant électronique :

---
{texte}
---

Écris UNE question précise, en français, à laquelle cet extrait permet de répondre (une valeur, une caractéristique, un comportement).
La question doit se comprendre seule : nomme le composant. Réponds uniquement par la question."""

random.seed(0)
echantillon = [c for c in random.sample(chunks, 40) if len(c["text"].split()) > 100][:20]

questions_synthetiques = []
for c in echantillon:
    reponse = completion(
        model=settings.GENERATION_MODEL,
        messages=[{"role": "user", "content": PROMPT_QUESTION.format(texte=c["text"])}],
        temperature=0, max_tokens=100,
    )
    question = reponse.choices[0].message.content.strip()
    questions_synthetiques.append({"question": question, "source": c["source"], "pages": [c["page"]]})
    print(f"- {question}  →  {c['source']} p.{c['page']}")

# Pour les ajouter au jeu de test après relecture :
# jeu_test += questions_synthetiques

### Bonus 6 — Évaluer la génération avec un LLM juge

Jusqu'ici, on a mesuré la recherche. Mais la réponse finale est-elle correcte ? C'est un texte libre : pas de
métrique simple. La pratique courante consiste à demander à un LLM de **juger**. On lui donne la question, le passage
source (la vérité) et la réponse du RAG ; il note de 0 à 10 et justifie sa note.

Gardez ses limites en tête : le juge peut se tromper, il a des biais (il aime les réponses longues) et chaque note
coûte un appel. On vérifie donc toujours un échantillon à la main. Ici, nous n'avons qu'un modèle : le juge est donc celui qui répond
(`settings.GENERATION_MODEL`). Idéalement, on choisit un juge au moins aussi bon que le modèle évalué : un modèle qui
juge ses propres réponses a tendance à les trouver bonnes. On note 8 questions.

In [ ]:
PROMPT_JUGE = """Tu évalues la réponse d'un assistant à une question technique.

Question : {question}

Passage de référence (contient la bonne réponse) :
---
{reference}
---

Réponse de l'assistant :
---
{reponse}
---

Note la réponse de 0 (fausse ou hors sujet) à 10 (exacte et complète), en te basant uniquement sur le passage de référence.
Réponds au format JSON : {{"note": <entier>, "justification": "<une phrase>"}}"""

pages_par_cle = {(p["source"], p["page"]): p["text"] for p in load_pdfs()}

notes = []
for item in jeu_test[:8]:
    reponse_rag, _ = answer(item["question"], chunks, embeddings)
    prompt = PROMPT_JUGE.format(question=item["question"],
                                reference=pages_par_cle[(item["source"], item["pages"][0])][:3000],
                                reponse=reponse_rag)
    reponse_juge = completion(model=settings.GENERATION_MODEL, messages=[{"role": "user", "content": prompt}],
                              temperature=0, max_tokens=150)
    verdict = json.loads(re.search(r"\{.*\}", reponse_juge.choices[0].message.content, re.S).group())  # extrait le JSON
    notes.append(verdict["note"])
    # ce que le juge a eu sous les yeux
    print(f"{verdict['note']:>2}/10  {item['question']}")
    print(f"       référence : {item['source']} p.{item['pages'][0]}")
    print(f"       RAG       : {reponse_rag[:300]}")
    print(f"       juge      : {verdict['justification']}\n")

print("note moyenne :", np.mean(notes))

**À observer**

- Lisez les justifications : êtes-vous d'accord avec le juge ? Sur quelles questions le RAG échoue-t-il, et pourquoi ?
  Mauvaise recherche, tableau mal lu, réponse trop prudente ?
- Cette note moyenne devient votre **indicateur** : changez le prompt, `k`, le modèle… et mesurez à nouveau.

### Bonus 7 — Un juge fiable ?

*À faire après le bonus 6, qui définit `PROMPT_JUGE` et `pages_par_cle`.*

La cellule est complète et s'exécute telle quelle. Elle fait juger **la même réponse** trois fois,
puis avec un passage de référence qui ne contient pas la réponse.
Les notes sont-elles identiques ? Le juge détecte-t-il la mauvaise référence ?
Concluez : quelle confiance accorder à une note isolée, et comment s'en servir quand même ?

In [ ]:
def juger(question, reference, reponse, model):
    # Reprend le juge du bonus 6, avec le modèle en paramètre
    prompt = PROMPT_JUGE.format(question=question, reference=reference, reponse=reponse)
    verdict = completion(model=model, messages=[{"role": "user", "content": prompt}], temperature=0, max_tokens=150)
    return json.loads(re.search(r"\{.*\}", verdict.choices[0].message.content, re.S).group())


item = jeu_test[0]
reference = pages_par_cle[(item["source"], item["pages"][0])][:3000]
reponse_rag, _ = answer(item["question"], chunks, embeddings)
print("question :", item["question"], "\n")

# 1. trois passages du même juge sur la même réponse
for essai in range(3):
    print("juge               :", juger(item["question"], reference, reponse_rag, settings.GENERATION_MODEL))

# 2. une référence qui ne contient pas la réponse
mauvaise_reference = pages_par_cle[("lm35.pdf", 1)][:3000]
print("mauvaise référence :", juger(item["question"], mauvaise_reference, reponse_rag, settings.GENERATION_MODEL))

### Bonus 8 — Les questions pièges

Un bon RAG doit dire qu'il ne sait pas quand la réponse n'est **pas** dans le corpus, plutôt qu'inventer.
Cinq questions pièges sont fournies. Complétez `refuse` puis mesurez le taux de refus corrects.

Comment détecter un refus ? Chercher des mots-clés est fragile ; demander au LLM est plus fiable, mais coûte un appel.
À vous de choisir. Ajoutez ensuite vos propres pièges : composant absent, information voisine mais absente
de la datasheet…

In [ ]:
questions_pieges = [
    "Quel est le prix du Raspberry Pi Pico ?",
    "Comment programmer l'ESP32 en MicroPython ?",
    "Quelle est la consommation du capteur DHT22 ?",                  # composant absent du corpus
    "Quelle est la température maximale de stockage du HC-SR04 ?",    # information absente de sa datasheet
    "Qui a inventé le transistor ?",
]


def refuse(reponse):
    # À compléter : renvoyer True si la réponse est un refus (information absente, "je ne sais pas"...).
    ...


# à décommenter une fois la fonction complétée :
# refus_corrects = 0
# for q in questions_pieges:
#     reponse, _ = answer(q, chunks, embeddings)
#     print("###", q, "\n" + reponse + "\n")
#     refus_corrects += bool(refuse(reponse))
# print(f"refus corrects : {refus_corrects}/{len(questions_pieges)}")

## Bilan

```
jeu de test (question, page)  ──►  retrieval : hit@k (+ MRR)   gratuit, rapide : à lancer à chaque changement
                              ──►  génération : LLM juge       coûteux : sur un échantillon (bonus 6)
```

Un RAG n'est jamais « fini » : chaque réglage se décide en mesurant sur un jeu de test représentatif.
C'est ce qui distingue une démo qui marche sur trois questions d'un système utilisable en production.